# AlzheimerCNN Ablation Study

This notebook is the ablation study for the `AlzheimerCNN` model contribution to PyHealth.

**Reference paper:** Liu et al. *"On the design of convolutional neural networks for automatic detection of Alzheimer's disease."* ML4H @ NeurIPS 2019. https://arxiv.org/abs/1911.03740

## Experimental Setup

We train five model configurations on the `Falah/Alzheimer_MRI` dataset and compare validation accuracy and macro-F1. All models share the same training recipe:

- **Optimizer:** AdamW, lr=1e-4, weight_decay=1e-5
- **Epochs:** 50
- **Batch size:** 32
- **Train/Val/Test split:** 60% / 20% / 20%
- **Loss:** Cross-entropy (4-class classification)

## Configurations Tested

| # | Model | Variation | Hypothesis |
|---|-------|-----------|------------|
| 1 | `AlzheimerCNN` | `init_channels=32` (baseline) | Reference point |
| 2 | `AlzheimerCNN` | `init_channels=16` | Smaller capacity may underfit |
| 3 | `AlzheimerCNN` | `init_channels=64` | Larger capacity may overfit on 5k samples |
| 4 | `AlzheimerCNNNormVariant` | `norm_type="group"` | GroupNorm may match InstanceNorm at small batch sizes |
| 4 | `AlzheimerCNNNormVariant` | `norm_type="layer"` | LayerNorm may match InstanceNorm at small batch sizes |
| 5 | `AlzheimerCNNViT` | CNN + Transformer hybrid | Self-attention may model global context better, but on small set may not be great |


## 1. Setup


In [12]:
import torch
import pandas as pd

from pyhealth.datasets import split_by_sample, get_dataloader
from pyhealth.trainer import Trainer
from pyhealth.models import (
    AlzheimerCNN,
    AlzheimerCNNNormVariant,
    AlzheimerCNNViT,
)

# ── Auto-detect device ──────────────────────────────────────────────────
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

print(f"Using device: {DEVICE}")

METRICS = ["accuracy", "f1_macro"]
EPOCHS = 50
BATCH_SIZE = 32
SEED = 42

torch.manual_seed(SEED)


Using device: mps


## 2. Load Dataset

Uses the `AlzheimersDataset` PyHealth-compatible wrapper around `Falah/Alzheimer_MRI` (4-class severity classification: NonDemented, VeryMildDemented, MildDemented, ModerateDemented).

The dataset class auto-downloads from HuggingFace and saves a `alzheimers-metadata.csv` for PyHealth's BaseDataset to consume.


In [13]:
from typing import Any, Dict, List
from pyhealth.tasks.base_task import BaseTask


class AlzheimersClassification(BaseTask):
    """Task for classifying Alzheimer's disease from brain images."""

    task_name: str = "AlzheimersClassification"
    input_schema: Dict = {"image": ("image", {"mode": "L", "image_size": 128})}
    output_schema: Dict[str, str] = {"label": "multiclass"}

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        events = patient.get_events(event_type="alzheimers")
        assert len(events) == 1, f"Expected 1 image, got {len(events)}"
        event = events[0]
        return [{"image": event.path, "label": event.label}]

In [17]:
import logging
import os
from pathlib import Path
from typing import Optional

import pandas as pd
from datasets import load_dataset
from pyhealth.datasets.base_dataset import BaseDataset

logger = logging.getLogger(__name__)


class AlzheimersDataset(BaseDataset):
    """PyHealth-compatible Alzheimer's dataset (auto-download + process)."""

    def __init__(
        self,
        root: str,
        dataset_name: Optional[str] = None,
        config_path: Optional[str] = None,
        cache_dir: Optional[str] = None,
        num_workers: int = 1,
        dev: bool = False,
        hf_dataset_name: str = "Falah/Alzheimer_MRI",
    ) -> None:
        self.hf_dataset_name = hf_dataset_name

        try:
            base_path = Path(__file__).parent
        except NameError:
            base_path = Path.cwd()

        if config_path is None:
            config_path = base_path / "configs" / "alzheimers.yaml"

        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        if not os.path.exists(metadata_path):
            self.prepare_metadata(root)

        super().__init__(
            root=root,
            tables=["alzheimers"],
            dataset_name=dataset_name or "alzheimers",
            config_path=config_path,
            cache_dir=cache_dir,
            num_workers=num_workers,
            dev=dev,
        )

    def prepare_metadata(self, root: str) -> None:
        os.makedirs(root, exist_ok=True)
        image_dir = os.path.join(root, "images")
        os.makedirs(image_dir, exist_ok=True)
        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        ds = load_dataset(self.hf_dataset_name, split="train")
        records = []
        for i, item in enumerate(ds):
            img_path = os.path.join(image_dir, f"{i}.png")
            if not os.path.isfile(img_path):
                item["image"].save(img_path)
            records.append({
                "patient_id": str(i),
                "path": img_path,
                "label": str(item["label"]),
            })
        pd.DataFrame(records).to_csv(metadata_path, index=False)

In [18]:
# Instantiate dataset and create train/val/test splits
dataset = AlzheimersDataset(root="Downloads")
dataset_samples = dataset.set_task(AlzheimersClassification(), num_workers=1)

train_samples, val_samples, test_samples = split_by_sample(
    dataset_samples, [0.6, 0.2, 0.2]
)

train_loader = get_dataloader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_samples,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_samples,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Initializing alzheimers dataset from Downloads (dev mode: False)
No cache_dir provided. Using default cache dir: /Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564
Setting task AlzheimersClassification for alzheimers base dataset...
Task cache paths: task_df=/Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/task_df.ld, samples=/Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: /Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/global_event_df.parquet
Scanning table: alzheimers from /Users/Joey/DL4H_Final_Project/PyHealth/examples/Downloads/alzheimers-metadata.csv
Caching event dataframe to /Users/Joey/Library/Caches/pyhe

  0%|          | 0/5120 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 5120/5120 [00:00<00:00, 7150.71it/s]

Worker 0 finished processing patients.


Fitting processors on the dataset...
Label label vocab: {'0': 0, '1': 1, '2': 2, '3': 3}
Processing samples and saving to /Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 5120 samples. (0 to 5120)


  0%|          | 0/5120 [00:00<?, ?it/s]

Rank 0 inferred the following `['tensor', 'tensor']` data format.


100%|██████████| 5120/5120 [00:01<00:00, 3188.99it/s]

Worker 0 finished processing samples.
Cached processed samples to /Users/Joey/Library/Caches/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Train: 3072 | Val: 1024 | Test: 1024


## 3. Training Helper

A single function that trains any model with the shared recipe and returns final validation metrics. This keeps every experiment cell short and ensures all configurations are compared fairly.


In [19]:
def train_and_evaluate(model, name: str) -> dict:
    """Train a model and return its final validation metrics.

    Args:
        model: An instantiated PyHealth model inheriting from BaseModel.
        name: Human-readable label for the configuration.

    Returns:
        Dict with keys 'name', 'accuracy', 'f1_macro', 'loss', 'params'.
    """
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    trainer = Trainer(model=model, metrics=METRICS, device=DEVICE)
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
    )

    scores = trainer.evaluate(val_loader)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "name": name,
        "accuracy": scores.get("accuracy", float("nan")),
        "f1_macro": scores.get("f1_macro", float("nan")),
        "loss": scores.get("loss", float("nan")),
        "params": num_params,
    }


# Collect all results in this list for the final summary
results = []


## 4. Experiments


### Experiment 1: Baseline (`init_channels=32`)


In [20]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=32, baseline)"))



Training: CNN (init_channels=32, baseline)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kerne

Epoch 0 / 50: 100%|██████████| 96/96 [00:05<00:00, 16.45it/s]

--- Train epoch-0, step-96 ---
loss: 1.1154



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 56.78it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0463



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:04<00:00, 19.49it/s]

--- Train epoch-49, step-4800 ---
loss: 0.5127



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 66.14it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.6582
f1_macro: 0.3370
loss: 0.7377



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 72.08it/s]


### Experiment 2: Smaller capacity (`init_channels=16`)


In [21]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=16)"))



Training: CNN (init_channels=16)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, st

Epoch 0 / 50: 100%|██████████| 96/96 [00:03<00:00, 31.44it/s]

--- Train epoch-0, step-96 ---
loss: 1.1758



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 103.97it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0692



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:02<00:00, 37.67it/s]

--- Train epoch-49, step-4800 ---
loss: 0.8010



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 133.27it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.5957
f1_macro: 0.3237
loss: 0.8656



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 137.44it/s]


### Experiment 3: Larger capacity (`init_channels=64`)


In [22]:
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=64,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (init_channels=64)"))



Training: CNN (init_channels=64)
AlzheimerCNN(
  (block1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=

Epoch 0 / 50: 100%|██████████| 96/96 [00:11<00:00,  8.48it/s]

--- Train epoch-0, step-96 ---
loss: 1.0862



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 32.73it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0358



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:10<00:00,  8.83it/s]

--- Train epoch-49, step-4800 ---
loss: 0.2423



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 34.51it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.8242
f1_macro: 0.5753
loss: 0.4499



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 33.80it/s]


### Experiment 4: Group normalization


In [23]:
model = AlzheimerCNNNormVariant(
    dataset=train_samples,
    norm_type="group",
    num_groups=8,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (norm_type=group)"))



Training: CNN (norm_type=group)
AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3)

Epoch 0 / 50: 100%|██████████| 96/96 [00:05<00:00, 17.40it/s]

--- Train epoch-0, step-96 ---
loss: 1.0931



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 50.32it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0470



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:05<00:00, 17.79it/s]

--- Train epoch-49, step-4800 ---
loss: 0.8400



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 49.43it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.5811
f1_macro: 0.3114
loss: 0.8598



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 48.81it/s]


### Experiment 5: Layer normalization


In [26]:
model = AlzheimerCNNNormVariant(
    dataset=train_samples,
    norm_type="layer",
    num_groups=8,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)
results.append(train_and_evaluate(model, "CNN (norm_type=layer)"))


Training: CNN (norm_type=layer)
AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(1, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3)

Epoch 0 / 50: 100%|██████████| 96/96 [00:06<00:00, 14.81it/s]

--- Train epoch-0, step-96 ---
loss: 1.0906



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 50.39it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0507



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:05<00:00, 17.51it/s]

--- Train epoch-49, step-4800 ---
loss: 0.8495



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 49.36it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.5645
f1_macro: 0.3068
loss: 0.8831



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 39.42it/s]


### Experiment 5: CNN + ViT hybrid


In [24]:
model = AlzheimerCNNViT(
    dataset=train_samples,
    init_channels=32,
    embed_dim=128,
    num_heads=4,
    num_transformer_layers=4,
    dropout=0.1,
)
results.append(train_and_evaluate(model, "CNN+ViT hybrid"))



Training: CNN+ViT hybrid
AlzheimerCNNViT(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (patch_embed): PatchEmbedding(
    (projection): Linear(in_features=1024, out_features=128, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamica

Epoch 0 / 50: 100%|██████████| 96/96 [00:05<00:00, 18.10it/s]

--- Train epoch-0, step-96 ---
loss: 1.0862



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 71.17it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.4990
f1_macro: 0.1664
loss: 1.0478



...


Epoch 49 / 50: 100%|██████████| 96/96 [00:04<00:00, 19.43it/s]

--- Train epoch-49, step-4800 ---
loss: 0.0530



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 70.83it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.8105
f1_macro: 0.6723
loss: 0.9272



Evaluation: 100%|██████████| 32/32 [00:00<00:00, 70.47it/s]


## 5. Results Summary

All six configurations side by side, sorted by validation accuracy.


In [27]:
df = pd.DataFrame(results)
df = df[["name", "params", "accuracy", "f1_macro", "loss"]]
df = df.sort_values("accuracy", ascending=False).reset_index(drop=True)

# Format columns for readability
df["params"]   = df["params"].apply(lambda x: f"{x:,}")
df["accuracy"] = df["accuracy"].apply(lambda x: f"{x:.4f}")
df["f1_macro"] = df["f1_macro"].apply(lambda x: f"{x:.4f}")
df["loss"]     = df["loss"].apply(lambda x: f"{x:.4f}")

print(df.to_string(index=False))


                            name    params accuracy f1_macro   loss
          CNN (init_channels=64) 1,616,004   0.8242   0.5753 0.4499
                  CNN+ViT hybrid   968,580   0.8105   0.6723 0.9272
CNN (init_channels=32, baseline)   421,252   0.6582   0.3370 0.7377
          CNN (init_channels=16)   114,180   0.5957   0.3237 0.8656
           CNN (norm_type=group)   422,212   0.5811   0.3114 0.8598
           CNN (norm_type=layer)   422,212   0.5645   0.3068 0.8831


## 6. Findings & Discussion

### Headline result


### Architecture finding: CNN beats CNN+ViT


### Normalization finding: Instance ≈ Group

### Normalization finding: Instance ≈ Layer 

### Practical recommendation

